# S&P 500 Custom Screener

Neutralny screener spółek z S&P 500. Parametry można dowolnie edytować na górze. Uruchom Runtime > Run all.

In [ ]:
# S&P 500 Custom Screener - Colab Ready
!pip install yfinance pandas tqdm openpyxl requests beautifulsoup4 lxml -q

import yfinance as yf
import pandas as pd
from tqdm import tqdm
import warnings
import numpy as np
warnings.filterwarnings('ignore')

# === PARAMETRY DO EDYCJI ===
MIN_MARKET_CAP_B = 10          # Minimalna kapitalizacja w mld USD
MAX_PE = 35                     # Maksymalne P/E
MIN_ROE = 0.12                  # Minimalne ROE (12%)
MIN_PROFIT_MARGIN = 0.10        # Minimalna marża zysku (10%)
MIN_REVENUE_GROWTH = 0.08       # Minimalny wzrost przychodów r/r
MAX_DEBT_EQUITY = 1.2           # Maksymalny Debt/Equity
MIN_ADX = 25                    # Minimalna siła trendu ADX
RSI_MAX = 65                    # RSI poniżej tej wartości
MIN_VOLUME_INCREASE = 30        # Minimalny wzrost volume vs średnia (%)
WEIGHT_FUND = 0.55
WEIGHT_TECH = 0.45

def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    tables = pd.read_html(url, header=0)
    return tables[0]['Symbol'].tolist()

def get_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    return true_range.rolling(period).mean()

def analyze_stock(ticker):
    try:
        t = yf.Ticker(ticker)
        info = t.info
        hist = t.history(period='6mo', auto_adjust=True)
        if len(hist) < 60: return None
        cap = info.get('marketCap', 0) or 0
        pe = info.get('trailingPE') or info.get('forwardPE') or 999
        roe = info.get('returnOnEquity') or 0
        margin = info.get('profitMargins') or 0
        rev_growth = info.get('revenueGrowth') or 0
        de = info.get('debtToEquity') or 999
        hist['vol_ma'] = hist['Volume'].rolling(20).mean()
        vol_inc = ((hist['Volume'].iloc[-1] / hist['vol_ma'].iloc[-1]) - 1) * 100
        delta = hist['Close'].diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rsi = 100 - (100 / (1 + gain/loss)).iloc[-1]
        atr = get_atr(hist)
        upper = (hist['High'] + hist['Low']).rolling(10).mean() + 3*atr
        in_uptrend = hist['Close'].iloc[-1] > upper.iloc[-1]
        f_score = 0
        if cap > MIN_MARKET_CAP_B*1e9: f_score += 20
        if 0 < pe < MAX_PE: f_score += 15
        if roe > MIN_ROE: f_score += 15
        if margin > MIN_PROFIT_MARGIN: f_score += 10
        if rev_growth > MIN_REVENUE_GROWTH: f_score += 10
        if de < MAX_DEBT_EQUITY: f_score += 10
        t_score = 0
        if vol_inc > MIN_VOLUME_INCREASE: t_score += 25
        if in_uptrend: t_score += 25
        if rsi < RSI_MAX: t_score += 20
        total = f_score * WEIGHT_FUND + t_score * WEIGHT_TECH
        grade = 'A' if total > 75 else 'B' if total > 60 else 'C' if total > 45 else 'D' if total > 30 else 'F'
        return {'Ticker': ticker, 'Name': info.get('shortName',''), 'Cap_B': round(cap/1e9,1), 'PE': round(pe,1) if pe<999 else None, 'ROE': round(roe*100,1), 'Score': round(total,1), 'Grade': grade}
    except: return None

tickers = get_sp500_tickers()
data = []
print('Scanning S&P 500...')
for tk in tqdm(tickers):
    res = analyze_stock(tk)
    if res: data.append(res)
df = pd.DataFrame(data).sort_values('Score', ascending=False)
print(df.head(20).to_string(index=False))
df.to_excel('SP500_Screener_Results.xlsx', index=False)
print('Done. Excel file ready to download.')